# Model Evaluation, Error Analysis, and Interpretation

This notebook reports the saved final test metrics and performs detailed subgroup, interpretability, crowd-level, and interval diagnostics on the validation partition. It does not use or recompute test target values. Validation informed tuning and model selection, so the detailed findings below are descriptive rather than a new independent estimate.


## Run the reusable evaluation workflow

Analysis code lives in `scripts/evaluate_final_model.py` and reusable metrics live in `src/interpretability.py`. This keeps the notebook focused on the results instead of duplicating project logic. The workflow derives crowd-level cut-points only from training targets.


In [ ]:
from PIL import Image

from src.config import (
    EVALUATION_SUMMARY_PATH,
    EVALUATION_ERROR_FIGURE_PATH,
    EVALUATION_LEVEL_FIGURE_PATH,
    EVALUATION_IMPORTANCE_FIGURE_PATH,
    EVALUATION_SHAP_SUMMARY_FIGURE_PATH,
    EVALUATION_SHAP_HOUR_FIGURE_PATH,
    EVALUATION_SHAP_TEMP_FIGURE_PATH,
    EVALUATION_SHAP_LOCAL_FIGURE_PATHS,
)
from scripts.evaluate_final_model import run_evaluation

results = run_evaluation()


## Real-world meaning and final test result

MAE is measured in people per observation. The report compares the final test MAE with the training-period mean count, not with gym capacity (capacity is not recorded). The final test metrics below are read from the saved metadata generated during the one permitted test evaluation.


In [ ]:
print(EVALUATION_SUMMARY_PATH.read_text(encoding='utf-8').split('## Crowd levels')[0])


## Error analysis by time, status, and crowd level

The subgroup charts show MAE by hour, day of week, holiday and semester flags, and actual crowd level. The complete table includes subgroup support, mean prediction bias, and the matching training-group mean so possible time-period shifts can be checked. Groups with limited observations should be interpreted cautiously.


In [ ]:
print(results['error_tables'].to_string(index=False))
Image.open(EVALUATION_ERROR_FIGURE_PATH)


### User-friendly Low / Medium / High accuracy

Both actual and predicted counts use the training-derived tercile thresholds saved in model metadata. This exact-match classification score is secondary: crossing a boundary by one person is counted as incorrect even when the numerical error is small.


In [ ]:
print('Training-derived thresholds:', results['thresholds'])
print(f"Validation level accuracy: {results['level_accuracy']:.1%}")
Image.open(EVALUATION_LEVEL_FIGURE_PATH)


## Permutation importance

Permutation importance shuffles one engineered feature at a time and measures how much validation MAE worsens. A larger increase means the fitted pipeline relies more on that input. Correlated features can share or mask importance, so this is model reliance rather than causation.


In [ ]:
print(results['importance'].head(15).to_string(index=False))
Image.open(EVALUATION_IMPORTANCE_FIGURE_PATH)


## SHAP global and dependence explanations

TreeSHAP decomposes each tree prediction into feature contributions relative to a baseline prediction. The summary and dependence plots use a deterministic validation sample. They describe the selected model's learned associations, not real-world causal effects.


In [ ]:
print(results['shap_importance'].head(15).to_string(index=False))
Image.open(EVALUATION_SHAP_SUMMARY_FIGURE_PATH)


In [ ]:
Image.open(EVALUATION_SHAP_HOUR_FIGURE_PATH)


In [ ]:
Image.open(EVALUATION_SHAP_TEMP_FIGURE_PATH)


### Three local predictions

The examples are selected from low, middle, and high parts of the validation prediction distribution. Each waterfall plot shows how encoded feature contributions move that individual prediction around the model baseline.


In [ ]:
print(results['local_rows'])
Image.open(EVALUATION_SHAP_LOCAL_FIGURE_PATHS[0])


The second local contribution plot follows.


In [ ]:
Image.open(EVALUATION_SHAP_LOCAL_FIGURE_PATHS[1])


The third local contribution plot follows.


In [ ]:
Image.open(EVALUATION_SHAP_LOCAL_FIGURE_PATHS[2])


## Prediction interval calibration

The 80% interval uses the 10th and 90th percentiles of training-only chronological out-of-fold residuals. Coverage is checked on validation data, not test data, to preserve the exact-once holdout rule. The measured coverage and average width are recorded in the report and interval artifact.


In [ ]:
print(results['interval'])


## Key findings and limitations

The generated `reports/evaluation_summary.md` contains the three worst supported error slices, evidence checks for their hypotheses, interpretation conclusions tied to saved importance figures, interval coverage, and the explicit **What the model cannot do** and limitations sections. In brief, the model cannot anticipate unrecorded events or use live crowd counts, and its learned associations do not establish causes. This one-gym observational dataset spans about 19 months and may reflect logging or collection conventions that do not generalize to other campuses or years.
